In [4]:
# ==========================================
# 0. CONNECT TO GOOGLE DRIVE
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import os
import json

# ==========================================
# 1. CONFIGURATION
# ==========================================
DATASET_PATH = "/content/drive/MyDrive/chest_xray"

MODEL_SAVE_NAME = "/content/drive/MyDrive/chest_model.h5"

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
EPOCHS = 10

print("Loading dataset...")

# ==========================================
# 2. DATA LOADING (FIXED)
# ==========================================
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "train"),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "val"),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# ==========================================
# 3. AUTO DETECT CLASSES
# ==========================================
NUM_CLASSES = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())

print(f"Detected Classes: {class_names}")
print(f"Number of Classes: {NUM_CLASSES}")

# ==========================================
# 4. BUILD MODEL (MobileNetV2)
# ==========================================
print("Building model...")

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)

outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)

# ==========================================
# 5. COMPILE
# ==========================================
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ==========================================
# 6. TRAIN
# ==========================================
print("Training started...")

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator
)

# ==========================================
# 7. SAVE MODEL
# ==========================================
model.save(MODEL_SAVE_NAME)

# Save class mapping
class_map_path = MODEL_SAVE_NAME.replace(".h5", "_classes.json")

with open(class_map_path, "w") as f:
    json.dump(train_generator.class_indices, f)

print("Model saved at:", MODEL_SAVE_NAME)
print("Class mapping saved at:", class_map_path)

print("✅ Training complete successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading dataset...
Found 9416 images belonging to 3 classes.
Found 4074 images belonging to 3 classes.
Detected Classes: ['normal', 'pneumonia', 'tuberculosis']
Number of Classes: 3
Building model...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Training started...
Epoch 1/10
295/295 ━━━━━━━━━━━━━━━━━━━━ 5203s 18s/step - accuracy: 0.9469 - loss: 0.1439 - val_accuracy: 0.9924 - val_loss: 0.0227
Epoch 2/10
295/295 ━━━━━━━━━━━━━━━━━━━━ 159s 540ms/step - accuracy: 0.9771 - loss: 0.0633 - val_accuracy: 0.9980 - val_loss: 0.0071
Epoch 3/10
295/295 ━━━━━━━━━━━━━━━━━━━━ 153s 519ms/step - accuracy: 0.9845 - loss: 0.0457 - val_accuracy: 0.9990 - val_loss: 0.0050
Epoch 4/10
295/295 ━━━━━━━━━━━━━━━━━━━━ 153s 518ms/step - accuracy: 0.9829 - loss: 0.0450 - val_accuracy: 0.9990 - val_loss: 0.0028
Epoch 5/10
295/295 ━━━━━━━━━━━━━━━━━━━━ 149s 506ms/step - accuracy: 0.9892 - loss: 0.0282 - val_accuracy: 0.9958 - val_loss: 0.0093
Epoch 6/10
295/295 ━━━━━━━━━━━━━━━━━━━━ 199s 495ms/step - accuracy: 0.9917 - loss: 0.0243 - val_accuracy: 0.9995 - val_loss: 0.0015
Epoch 7/10
295/295 ━━━━━━━━━━━━━━━━━━━━ 149s 506ms/step - accuracy: 0.9900 - loss: 0.0232 - val_accuracy: 0.9990 - val_loss: 0.0041
Epoch 8/10
295/295 ━━━━━━━━━━━━━━━━━━━━ 146s 497ms/step -

Model saved at: /content/drive/MyDrive/chest_model.h5
Class mapping saved at: /content/drive/MyDrive/chest_model_classes.json
✅ Training complete successfully!
